# Fine-Tuning XLM-RoBERTa untuk Disease Classification
**Disease Surveillance AI** — Google Colab T4 GPU

1. Upload `train.jsonl` dan `test.jsonl` ke Google Drive
2. Jalankan cell di bawah satu per satu
3. Download model dari Drive setelah selesai

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets evaluate sentencepiece accelerate

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Load dataset
import json
from datasets import Dataset

BASE = "/content/drive/MyDrive/disease-nlp"

with open(f"{BASE}/train.jsonl") as f:
    train_data = [json.loads(line) for line in f if line.strip()]
with open(f"{BASE}/test.jsonl") as f:
    test_data = [json.loads(line) for line in f if line.strip()]

print(f"Train: {len(train_data)} samples")
print(f"Test:  {len(test_data)} samples")

In [ ]:
# Cell 4: Labels
# Sync dengan database nlp_labels / config.py
LABELS = [
    'dengue fever DBD', 'acute diarrhea', 'leptospirosis',
    'influenza flu', 'COVID-19 coronavirus', 'malaria',
    'tuberculosis TB', 'chikungunya', 'pneumonia',
    'typhoid fever', 'measles campak',
    'hantavirus', 'coronavirus MERS',
    'NEGATIVE - not health related',
]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

# Filter: hanya ambil data yang labelnya ada di LABELS
train_data = [d for d in train_data if d['disease'] in label2id]
test_data = [d for d in test_data if d['disease'] in label2id]
print(f"After filter: train={len(train_data)} test={len(test_data)}")

# Convert to Dataset
train_ds = Dataset.from_list(train_data).map(lambda x: {'labels': label2id[x['disease']]})
test_ds = Dataset.from_list(test_data).map(lambda x: {'labels': label2id[x['disease']]})


In [ ]:
# Cell 5: Load model + tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "xlm-roberta-base"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS),
    id2label=id2label, label2id=label2id,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

print(f"Model loaded: {MODEL_NAME}")

In [ ]:
# Cell 6: Training
from transformers import TrainingArguments, Trainer
from evaluate import load
import numpy as np

accuracy = load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return accuracy.compute(predictions=preds, references=p.label_ids)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Cell 7: Evaluation
eval_result = trainer.evaluate()
print(f"\nAccuracy: {eval_result['eval_accuracy']:.4f}")

# Per-class metrics
from sklearn.metrics import classification_report
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

In [ ]:
# Cell 8: Export model ke Google Drive
import shutil
import os

output_dir = f"{BASE}/model"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to: {output_dir}")

# Show size
total = sum(os.path.getsize(f"{output_dir}/{f}") for f in os.listdir(output_dir) if os.path.isfile(f"{output_dir}/{f}"))
print(f"Model size: {total / 1024 / 1024:.1f} MB")

In [ ]:
# Cell 9: Test prediction (opsional)
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=output_dir,
    tokenizer=output_dir,
    device=-1,
)

tests = [
    "50 warga Jakarta terkena DBD setelah banjir",
    "Pertandingan sepak bola Persija vs Persib",
    "25 pasien diare akut dirawat di Puskesmas Bogor",
]
for t in tests:
    r = classifier(t, return_all_scores=False)[0]
    print(f"  {t[:50]:50s} → {r['label']:30s} ({r['score']:.3f})")

## Deploy ke Server

Setelah model di-download:

```bash
# Di WSL
mkdir -p services/nlp-python/models/fine-tuned
# Copy file model ke folder tersebut
# Lalu:
docker compose up -d nlp-python
```